In [13]:
import pandas as pd
import csv
import numpy as np

# get df from csv (in local directory)
df = pd.read_csv(r'C:\Users\jeong\research\grading-data-6-17-22.csv')

# general tools

# returns filtered dataframe. Each condition should be passed as column name = LIST of targets
    # e.g. "filter(df, crsTitle = ['PHYSICS I LAB'], facultyID = ['F18125', 'F97128'])" returns df with 84 rows
def filter(df, **kwargs):
    for key in kwargs.keys():
        df = df[(df[key]).isin(kwargs.get(key))]
    return df

# check if object is nan
def isNaN(string):
    return string != string

# returns a dictionary of "facultyID: number of unique CRNs attributed to this instructor"
def total_courses_per_instructor(df):
    faculty_list = np.unique(list(df['facultyID']))
    faculty_courses = { i : 0 for i in faculty_list }
    for i in faculty_list:
        faculty_history = filter(df, facultyID = [i])
        faculty_courses.update({i: (len(np.unique(list((faculty_history)['CRN']))))})
    return faculty_courses


# data cleaning
    # original df remains unchanged. cleaned data is put in new dataframe final_df
cleaned_df = df.copy()
cleaned_df.replace(" ", np.nan, inplace=True)
cleaned_df['finGradN'] = cleaned_df['finGradN'].astype('float')
cleaned_df['facultyID']=cleaned_df['facultyID'].astype(str)
# drop administrative CBA/FCRH as they are not actual students
cleaned_df.drop(cleaned_df[cleaned_df['ProgCode'] == 'Administrative CBA'].index,  inplace = True)
cleaned_df.drop(cleaned_df[cleaned_df['ProgCode'] == 'Administrative FCRH'].index, inplace = True)
# creates a new CRN: CRN + last two digits of year + one digit based on semester
    # e.g. oldCRN 11135, Summer 2010 course -> CRN: 11135102
cleaned_df[['sem', 'year']] = cleaned_df['term'].str.split(' ', n = 1, expand = True)
cleaned_df['year'] = pd.to_numeric(cleaned_df['year'])
cleaned_df['year'] = cleaned_df.apply(lambda x: x['year'] % 1000, axis=1)
cleaned_df.loc[cleaned_df['sem'] == 'Fall', 'sem'] = 0
cleaned_df.loc[cleaned_df['sem'] == 'Spring', 'sem'] = 1
cleaned_df.loc[cleaned_df['sem'] == 'Summer', 'sem'] = 2
cleaned_df['sem'] = pd.to_numeric(cleaned_df['sem'])
cleaned_df.rename(columns = {'CRN':'oldCRN'}, inplace = True)
cleaned_df['oldCRN'] = cleaned_df['oldCRN'].astype(str)
cleaned_df['year'] = cleaned_df['year'].astype(str)
cleaned_df['sem'] = cleaned_df['sem'].astype(str)
cleaned_df['CRN'] = cleaned_df['oldCRN'] + cleaned_df['year']+ cleaned_df['sem']

final_df = cleaned_df.copy()
final_df['NumCode'] = final_df['NumCode'].fillna(0)
final_df['NumCode'] = final_df['NumCode'].astype(int)
final_df['NumCode'] = final_df['NumCode'].astype(str)
final_df['CourseCode'] = final_df['ProgCode'] + " " + final_df['NumCode']

# data is all cleaned and saved in dataframe final_df
final_df

,Unnamed: 0,SID,ProgCode,NumCode,School,oldCRN,credHrs,term,class_size,campus,...,major2,minor,minor2,degree,finGradC,finGradN,sem,year,CRN,CourseCode
0,8,S94511,Theatre,2751,Summer Session 2,11135,4.0,Summer 2010,1,Off-Site,...,NaN,NaN,NaN,NaN,A,4.00,2,10,11135102,Theatre 2751
1,14,S42486,Visual Arts,2700,Fordham College/Lincoln Ctr,11710,4.0,Spring 2011,8,Lincoln Center,...,NaN,NaN,NaN,NaN,F,0.00,1,11,11710111,Visual Arts 2700
2,16,S42706,Music,1261,Fordham College/Rose Hill,10797,0.0,Fall 2010,12,Rose Hill,...,NaN,NaN,NaN,NaN,F,0.00,0,10,10797100,Music 1261
3,17,S71751,Visual Arts,3500,Summer Session 2,10752,4.0,Summer 2010,10,Off-Site,...,NaN,NaN,NaN,NaN,A,4.00,2,10,10752102,Visual Arts 3500
4,19,S32275,Comm and Media Studies,1010,Fordham College/Lincoln Ctr,10603,3.0,Fall 2012,35,Lincoln Center,...,NaN,NaN,NaN,NaN,C-,1.67,0,12,10603120,Comm and Media Studies 1010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446503,473521,S59870,Visual Arts,1128,Summer Session 1,10134,4.0,Summer 2018,13,Rose Hill,...,NaN,NaN,NaN,NaN,C+,2.33,2,18,10134182,Visual Arts 1128
446504,473522,S63569,Visual Arts,1128,Summer Session 1,10134,4.0,Summer 2018,13,Rose Hill,...,NaN,NaN,NaN,NaN,A-,3.67,2,18,10134182,Visual Arts 1128
446505,473523,S94925,Visual Arts,1128,Summer Session 1,10134,4.0,Summer 2018,13,Rose Hill,...,NaN,NaN,NaN,NaN,A,4.00,2,18,10134182,Visual Arts 1128
446506,473524,S32156,Visual Arts,1128,Summer Session 1,10134,4.0,Summer 2018,13,Rose Hill,...,NaN,NaN,NaN,NaN,B,3.00,2,18,10134182,Visual Arts 1128


In [19]:
# returns Average, SD, Z-score of GPA of one section, for a particular CourseCode course_name, for sections with more than class_size students
    # e.g. gpa_by_section(final_df, 'Mathematics 2006', 6) returns a dataframe of columns: CRN, Instructor, GPA, SD, Class Size, Z-score
    # class size was separately computed. (did not use class_size column in original df as it was inaccurate.)
def gpa_by_section(df, course_name, class_size):
    sections = list(np.unique((filter(df, CourseCode = [course_name])['CRN'])))
    to_return = pd.DataFrame(columns =['CRN', 'Instructor', 'GPA', 'SD', 'Class Size'])
    
    for i in sections:
        filtered_data = filter(df, CourseCode = [course_name], CRN = [i])
        to_return.loc[len(to_return)] = [i, (filtered_data.mode()['facultyID'][0]), np.mean(((filtered_data)['finGradN']).to_numpy()), np.std(((filtered_data)['finGradN']).to_numpy()), len(filtered_data.index)]

    to_return = to_return[to_return['Class Size'] >= class_size]

    ttl_mean = (np.mean(to_return.GPA.values.tolist()))
    ttl_sd = (np.std(to_return.GPA.values.tolist()))
    to_return['Z-score'] = (to_return['GPA']-ttl_mean)/ttl_sd

    return to_return

# creates list of course names in order of highest occurences in dataframe
course_data_points = {}
for i in list(final_df['CourseCode']):
  course_data_points[i] = course_data_points.get(i, 0) + 1
sorted_tuples = sorted(course_data_points.items(), key=lambda item: item[1], reverse=True)
courses_in_order = list(({k: v for k, v in sorted_tuples}).keys())

In [31]:
# parameters: modify as needed.
target_course_list = courses_in_order[:100]     # which master list of courses to go through
z_score_cutoff = 1.0    # Z-score positively and negatively to flag section
flag_percentage = 70    # Percentage of sections that the instructor taught that need to be flagged for final list result
instructor_low_limit = 5    # low limit of number of sections the instructor should have teached to show up in final result flagged_easy and flagged_hard
instructor_high_limit = 100 # high limit of number of sections the instructor should have teached to show up in final result flagged_easy and flagged_hard

In [32]:
# list of unique instructors (facultyID) / courses (CourseCode)
instructors = [k for (k,v) in total_courses_per_instructor(final_df).items() if instructor_low_limit <= v <= instructor_high_limit]
courses = np.unique(list(final_df['CourseCode']))

# loops through every course and adds +1 to the value of the instructor in appropriate dictionary if a section they taught had exceptionally high/low grades
# then, prints out instructors who were flagged for more than a certain percentage of the courses they taught

# run time tracker
current_index = 0
total_index = len(target_course_list)

# dictionary of instructors that taught sections with exceptionally high/low grades
easy_instructor_tally = { i : 0 for i in instructors }
hard_instructor_tally = { i : 0 for i in instructors }


for i in target_course_list:
    # disregards sections with class size less than two standard deviations below the average class size of that course
    all_class_sizes = (list(gpa_by_section(final_df, i, 0)['Class Size']))
    all_sections = gpa_by_section(final_df, i, (np.mean(all_class_sizes))-2*(np.std(all_class_sizes)))

    # "exceptionally low or high" defined as z_score_cutoff standard deviations
    easy_anomalies = all_sections[all_sections['Z-score'] > z_score_cutoff]
    if len(easy_anomalies.index) > 0 : 
        for j in list(easy_anomalies['Instructor']):
            if not (isNaN(j)):
                if j in instructors:
                    easy_instructor_tally[j]+=1

    hard_anomalies = all_sections[all_sections['Z-score'] < -z_score_cutoff]
    if len(hard_anomalies.index) > 0 : 
        for k in list(hard_anomalies['Instructor']):
            if not (isNaN(k)):
                if k in instructors:
                    hard_instructor_tally[k]+=1

    # run time tracker
    current_index+=1
    print("current index: "+str(current_index)+" out of "+str(total_index))

num_courses_taught_per_instructor = total_courses_per_instructor(final_df)

flagged_easy = [k for k, v in easy_instructor_tally.items() if v > (num_courses_taught_per_instructor[k])*(flag_percentage/100)]
flagged_hard = [k for k, v in hard_instructor_tally.items() if v > (num_courses_taught_per_instructor[k])*(flag_percentage/100)]

print(flagged_easy)
print(flagged_hard)

current index: 1 out of 100
current index: 2 out of 100
current index: 3 out of 100
current index: 4 out of 100
current index: 5 out of 100
current index: 6 out of 100
current index: 7 out of 100
current index: 8 out of 100
current index: 9 out of 100
current index: 10 out of 100
current index: 11 out of 100
current index: 12 out of 100
current index: 13 out of 100
current index: 14 out of 100
current index: 15 out of 100
current index: 16 out of 100
current index: 17 out of 100
current index: 18 out of 100
current index: 19 out of 100
current index: 20 out of 100
current index: 21 out of 100
current index: 22 out of 100
current index: 23 out of 100
current index: 24 out of 100
current index: 25 out of 100
current index: 26 out of 100
current index: 27 out of 100
current index: 28 out of 100
current index: 29 out of 100
current index: 30 out of 100
current index: 31 out of 100
current index: 32 out of 100
current index: 33 out of 100
current index: 34 out of 100
current index: 35 out o